# 02 — GQA and deeper-model comparison

This notebook benchmarks two Grouped Query Attention configurations with tied input/output embeddings:

1. **GQA-13L:** the same 13-layer depth as the MHA baseline, with 8 query heads and 2 K/V heads.
2. **GQA-15L:** a deeper model that uses the parameter budget released by GQA while remaining below the 50M limit.

Run `01_mha_baseline.ipynb` first to include measured MHA speed and memory in the final comparison. Synthetic tokens are used so the results measure systems performance rather than model quality. All runs use the same Weights & Biases project and group.

In [ ]:
from copy import deepcopy
import json
from pathlib import Path
import time

import torch
import torch.nn.functional as F
import wandb

from llm_mini_lab.models.gpt import GPTModel
from llm_mini_lab.training.core import GPT_CONFIG_50M

SEED = 42
BATCH_SIZE = 2
SEQUENCE_LENGTH = 128
WARMUP_STEPS = 2
TRAIN_STEPS = 10
LEARNING_RATE = 3e-4
WANDB_PROJECT = "50M-LLM-GQA"
WANDB_GROUP = "mha-vs-gqa-16k-rope"

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
print("Device:", device)
if device.type != "cuda":
    print("CUDA is not available: SDPA will run, but this run cannot verify a CUDA fused kernel or peak VRAM.")
wandb.login()


In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def run_training_trial(name, config):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = GPTModel(config).to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=name,
        config={**config, "batch_size": BATCH_SIZE, "sequence_length": SEQUENCE_LENGTH, "warmup_steps": WARMUP_STEPS, "train_steps": TRAIN_STEPS, "learning_rate": LEARNING_RATE},
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and amp_dtype == torch.float16)
    generator = torch.Generator(device=device).manual_seed(SEED)
    tokens = torch.randint(0, config["vocab_size"], (BATCH_SIZE, SEQUENCE_LENGTH + 1), device=device, generator=generator)

    def step():
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(tokens[:, :-1])
            loss = F.cross_entropy(logits.flatten(0, 1).float(), tokens[:, 1:].flatten())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        return loss.detach().item()

    for _ in range(WARMUP_STEPS):
        step()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    losses = []
    start = time.perf_counter()
    for step_index in range(TRAIN_STEPS):
        loss = step()
        losses.append(loss)
        run.log({"train/loss": loss}, step=step_index)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    result = {
        "name": name,
        "device": torch.cuda.get_device_name(0) if device.type == "cuda" else str(device),
        "parameters": count_parameters(model),
        "layers": config["n_layers"],
        "ff_hidden_dim": config.get("ff_hidden_dim", 4 * config["emb_dim"]),
        "query_heads": config["n_heads"],
        "kv_heads": config["n_kv_heads"],
        "batch_size": BATCH_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "steps": TRAIN_STEPS,
        "initial_loss": losses[0],
        "final_loss": losses[-1],
        "milliseconds_per_step": 1000 * elapsed / TRAIN_STEPS,
        "tokens_per_second": BATCH_SIZE * SEQUENCE_LENGTH * TRAIN_STEPS / elapsed,
        "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None,
    }
    run.summary.update(result)
    run.finish()
    del model, optimizer
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result


In [ ]:
common_overrides = {"vocab_size": 16_384, "tokenizer_name": "sp16384", "positional_encoding": "rope"}

mha_config = deepcopy(GPT_CONFIG_50M)
mha_config.update(common_overrides)
mha_config["n_layers"] = 13
mha_config["n_kv_heads"] = mha_config["n_heads"]

gqa_13l_config = deepcopy(GPT_CONFIG_50M)
gqa_13l_config.update(common_overrides)
gqa_13l_config["n_layers"] = 13
gqa_13l_config["n_kv_heads"] = 2

gqa_15l_config = deepcopy(gqa_13l_config)
gqa_15l_config["n_layers"] = 15

mha_budget = count_parameters(GPTModel(mha_config))
gqa_13l_parameters = count_parameters(GPTModel(gqa_13l_config))
gqa_15l_parameters = count_parameters(GPTModel(gqa_15l_config))

print(f"13-layer MHA: {mha_budget:,}")
print(f"13-layer GQA: {gqa_13l_parameters:,} ({mha_budget - gqa_13l_parameters:,} saved at equal depth)")
print(f"15-layer GQA: {gqa_15l_parameters:,} ({50_000_000 - gqa_15l_parameters:,} below the 50M limit)")
assert mha_budget < 50_000_000
assert gqa_15l_parameters < 50_000_000


In [ ]:
gqa_results = [
    run_training_trial("GQA-13L", gqa_13l_config),
    run_training_trial("GQA-15L", gqa_15l_config),
]
gqa_results


In [ ]:
results_dir = Path("results")
baseline_path = results_dir / "mha_baseline.json"
all_results = []
if baseline_path.exists():
    all_results.append(json.loads(baseline_path.read_text(encoding="utf-8")))
else:
    print("MHA result not found. Run 01_mha_baseline.ipynb first for measured MHA performance.")
all_results.extend(gqa_results)

columns = ["name", "parameters", "layers", "query_heads", "kv_heads", "milliseconds_per_step", "tokens_per_second", "peak_allocated_gb"]
print(" | ".join(columns))
for result in all_results:
    print(" | ".join(str(result.get(column)) for column in columns))

results_dir.mkdir(exist_ok=True)
result_path = results_dir / "gqa_comparison.json"
result_path.write_text(json.dumps(all_results, indent=2), encoding="utf-8")
print("Saved:", result_path.resolve())


## What to report

Please share the GPU model, PyTorch/CUDA versions, parameters, tokens/s, milliseconds per step, peak allocated VRAM, and whether the 15-layer run completed without instability. A longer run on real data with `tokenizers/fineweb_16384_bpe.model` is still required before drawing conclusions about model quality.